# DeepSeek V4 Compressed Sparse Attention From Scratch

This notebook implements the core idea behind DeepSeek V4-style Compressed
Sparse Attention (CSA) in a small CPU-friendly form.

Focus:

- compress token KV entries into block-level entries
- build indexer scores over compressed entries
- apply causal visibility for compressed blocks
- select top-k compressed blocks
- combine local token KV with selected compressed KV
- run sparse attention over that reduced set

Not included here: RoPE, quantization, Hadamard rotation, distributed context
parallelism, HCA, or custom kernels.

## Setup

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)


def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        for idx, value in enumerate(row):
            widths[idx] = max(widths[idx], len(str(value)))
    template = "  ".join(f"{{:<{width}}}" for width in widths)
    print(template.format(*headers))
    print(template.format(*["-" * width for width in widths]))
    for row in rows:
        print(template.format(*[str(value) for value in row]))

## 1. Dense Decode Read Baseline

Dense decode reads every previous KV entry. CSA reduces that read set by using:

```text
local recent tokens + top-k compressed historical blocks
```

In [ ]:
seq_len = 64
block_size = 8
top_k = 3
local_window = 6
batch = 1
dim = 16
index_heads = 2
index_dim = 8
query_heads = 4

num_blocks = math.ceil(seq_len / block_size)
rows = [
    ["sequence length", seq_len],
    ["block size", block_size],
    ["compressed blocks", num_blocks],
    ["top-k compressed blocks", top_k],
    ["local token window", local_window],
]
print_table(["quantity", "value"], rows)

## 2. Gated Block Compression

DeepSeek V4 CSA compresses groups of tokens into compressed KV entries. The
production model uses learned projections and gates. Here we implement the core
operation:

```text
compressed_block = sum softmax(gate_logits within block) * token_values
```

The softmax is over tokens inside each block, separately for each feature dim.

In [ ]:
def gated_block_compress(values, gate_logits, block_size):
    if values.shape != gate_logits.shape:
        raise ValueError("values and gate_logits must have same shape")
    batch, seq_len, dim = values.shape
    num_blocks = math.ceil(seq_len / block_size)
    padded_len = num_blocks * block_size
    pad_len = padded_len - seq_len

    if pad_len:
        values = F.pad(values, (0, 0, 0, pad_len), value=0)
        gate_logits = F.pad(gate_logits, (0, 0, 0, pad_len), value=float("-inf"))

    values_by_block = values.view(batch, num_blocks, block_size, dim)
    gates_by_block = gate_logits.view(batch, num_blocks, block_size, dim)
    weights = torch.softmax(gates_by_block, dim=2)
    compressed = (values_by_block * weights).sum(dim=2)
    return compressed, weights


token_values = torch.randn(batch, seq_len, dim)
gate_logits = torch.randn(batch, seq_len, dim)
compressed_kv, compression_weights = gated_block_compress(
    token_values,
    gate_logits,
    block_size,
)
print("token_values", list(token_values.shape))
print("compressed_kv", list(compressed_kv.shape))
print("compression_weights", list(compression_weights.shape))

## 3. Compressed Block Visibility

A compressed block should not be visible before its tokens exist. For prefill,
query token `q` can see compressed block `b` only after the block is complete.

In [ ]:
def compressed_block_visibility(query_len, num_blocks, block_size, device=None):
    query_positions = torch.arange(query_len, device=device)
    block_ids = torch.arange(num_blocks, device=device)
    visible_blocks = (query_positions + 1) // block_size
    return block_ids.unsqueeze(0) < visible_blocks.unsqueeze(1)


visible = compressed_block_visibility(seq_len, num_blocks, block_size)
print("visible mask", list(visible.shape))
print_table(
    ["query token", "visible compressed blocks"],
    [[q, visible[q].nonzero().flatten().tolist()] for q in [0, 7, 8, 15, 63]],
)

## 4. Lightning-Indexer-Style Scores

The DeepSeek inference code scores compressed entries with several index heads:

```text
score(q, block) = sum_h weight_h * ReLU(index_query_h dot compressed_key)
```

This is not normal attention yet. It is a routing/indexing stage that decides
which compressed entries core attention will read.

In [ ]:
def lightning_index_scores(index_queries, compressed_index_keys, head_weights):
    raw_scores = torch.einsum("bqhd,bnd->bqhn", index_queries, compressed_index_keys)
    return (raw_scores.relu() * head_weights.unsqueeze(-1)).sum(dim=2)


index_queries = torch.randn(batch, seq_len, index_heads, index_dim)
compressed_index_keys = torch.randn(batch, num_blocks, index_dim)
head_weights = torch.randn(batch, seq_len, index_heads)
index_scores = lightning_index_scores(index_queries, compressed_index_keys, head_weights)
print("index_scores", list(index_scores.shape))

## 5. Top-k Selection Under Causal Visibility

The indexer scores all compressed blocks, then causal masking removes invisible
future blocks. Top-k runs over the remaining visible blocks.

Invalid slots are represented as `-1`.

In [ ]:
def select_topk_compressed_blocks(index_scores, visible_mask, top_k):
    masked_scores = index_scores.masked_fill(~visible_mask.unsqueeze(0), float("-inf"))
    k = min(top_k, index_scores.shape[-1])
    top_scores, top_indices = masked_scores.topk(k, dim=-1)
    top_indices = torch.where(torch.isfinite(top_scores), top_indices, -1)
    if k == top_k:
        return top_indices
    pad = top_indices.new_full((*top_indices.shape[:2], top_k - k), -1)
    return torch.cat([top_indices, pad], dim=-1)


selected_blocks = select_topk_compressed_blocks(index_scores, visible, top_k)
print("selected_blocks", list(selected_blocks.shape))
print_table(
    ["query token", "selected compressed blocks"],
    [[q, selected_blocks[0, q].tolist()] for q in [0, 7, 8, 15, 63]],
)

## 6. Local Window Indices

CSA also keeps a local token window. This protects recent fine-grained detail
that block compression may blur.

In [ ]:
def local_token_indices(query_len, local_window, device=None):
    query_positions = torch.arange(query_len, device=device).unsqueeze(1)
    offsets = torch.arange(local_window, device=device).unsqueeze(0)
    indices = query_positions - local_window + 1 + offsets
    return torch.where(indices >= 0, indices, -1)


local_indices = local_token_indices(seq_len, local_window)
print("local_indices", list(local_indices.shape))
print_table(
    ["query token", "local token indices"],
    [[q, local_indices[q].tolist()] for q in [0, 3, 8, 63]],
)

## 7. Sparse Attention Over Local + Compressed Entries

For each query, gather:

```text
local token KV entries + selected compressed KV entries
```

Then run normal attention over that smaller set.

In [ ]:
def sparse_attention(query, token_kv, compressed_kv, local_indices, selected_blocks):
    batch, query_len, query_heads, dim = query.shape
    output = torch.empty_like(query)
    max_entries = local_indices.shape[1] + selected_blocks.shape[2]
    all_weights = query.new_zeros(batch, query_len, query_heads, max_entries)

    for b in range(batch):
        for q_pos in range(query_len):
            entries = []
            for token_idx in local_indices[q_pos].tolist():
                if 0 <= token_idx < token_kv.shape[1]:
                    entries.append(token_kv[b, token_idx])
            for block_idx in selected_blocks[b, q_pos].tolist():
                if 0 <= block_idx < compressed_kv.shape[1]:
                    entries.append(compressed_kv[b, block_idx])

            kv_entries = torch.stack(entries, dim=0)
            scores = query[b, q_pos] @ kv_entries.T / math.sqrt(dim)
            weights = torch.softmax(scores, dim=-1)
            output[b, q_pos] = weights @ kv_entries
            all_weights[b, q_pos, :, : kv_entries.shape[0]] = weights

    return output, all_weights


query = torch.randn(batch, seq_len, query_heads, dim)
sparse_out, sparse_weights = sparse_attention(
    query,
    token_values,
    compressed_kv,
    local_indices,
    selected_blocks,
)
print("sparse_out", list(sparse_out.shape))
print("sparse_weights", list(sparse_weights.shape))
print("weight row sums", sparse_weights.sum(dim=-1)[0, -1])

## 8. Dense Reads vs CSA Reads

This is the first payoff. Dense attention reads `seq_len` entries per query.
CSA reads roughly `local_window + top_k` entries once enough compressed blocks
exist.

In [ ]:
def csa_read_count_for_query(q_pos, block_size, top_k, local_window):
    visible_blocks = (q_pos + 1) // block_size
    selected = min(top_k, visible_blocks)
    local = min(local_window, q_pos + 1)
    return selected + local


positions = torch.arange(seq_len)
dense_reads = positions + 1
csa_reads = torch.tensor(
    [csa_read_count_for_query(int(q), block_size, top_k, local_window) for q in positions]
)

plt.figure(figsize=(8, 4.5))
plt.plot(positions, dense_reads, label="dense reads")
plt.plot(positions, csa_reads, label="CSA reads")
plt.xlabel("query position")
plt.ylabel("entries read")
plt.title("Dense vs compressed sparse reads")
plt.legend()
plt.show()

print_table(
    ["query", "dense", "CSA", "reduction"],
    [
        [q, int(dense_reads[q]), int(csa_reads[q]), f"{dense_reads[q] / csa_reads[q]:.1f}x"]
        for q in [7, 15, 31, 63]
    ],
)

## 9. Selection Pattern Visualization

Rows are query positions. Columns are compressed block IDs. A filled cell means
that block was selected by top-k routing.

In [ ]:
selection_matrix = torch.zeros(seq_len, num_blocks)
for q in range(seq_len):
    for block_idx in selected_blocks[0, q].tolist():
        if block_idx >= 0:
            selection_matrix[q, block_idx] = 1

plt.figure(figsize=(7, 5))
plt.imshow(selection_matrix, aspect="auto", interpolation="nearest", cmap="Greens")
plt.xlabel("compressed block")
plt.ylabel("query position")
plt.title("Top-k compressed block selection")
plt.show()

## 10. What Moves Into Package Code

The reusable implementation should expose:

- gated block compression
- compressed block visibility
- index score computation
- top-k selection
- local token index generation
- sparse attention over local + compressed entries

That is exactly the package boundary for the first reference version.